# Bisaya Kaldi HMM-GMM Training

Trains a 2-gram, 3-state, speaker-adaptive-trained (SAT) HMM-GMM ASR model
for Bisaya using the PS27 27-phoneme set. Reproduces the best-performing
Bisaya configuration from Ing (2023) -- every neural (DNN/TDNN) variant
that thesis tested for Bisaya scored equal to or worse than this HMM-GMM
setup (5.41% WER in the thesis). See Section 1 for the per-decision
rationale.

**Environment:** requires WSL2/Ubuntu (Kaldi does not build on native
Windows). Assumes an existing Kaldi checkout (`KALDI_ROOT`, default
`~/kaldi`) -- this notebook never clones Kaldi itself. `CORPUS_DIR` and
`RESULTS_DIR` stay Windows-hosted (via WSL's `/mnt/...` bind mount);
`WORK_DIR` (Kaldi build artifacts, `data/`, `mfcc/`, `exp/`) lives on the
WSL-native filesystem, since Kaldi's build/training stages create very
large numbers of small files -- much slower over the Windows bind mount
than on native ext4.

Every stage is wrapped in `stage(name, done, fn)`, which skips work whose
output already exists -- safe to re-run top to bottom after an
interruption.


## 1. Decisions Made Where the Paper/Guide Leaves Things Unspecified

| Decision | Choice | Rationale |
|---|---|---|
| Model architecture | HMM-GMM (no DNN/TDNN stage) | The thesis found every neural (DNN/TDNN) variant it tested for Bisaya scored equal to or worse than HMM-GMM. |
| Model enhancement | SAT (speaker-adaptive training) | Best-performing enhancement in the thesis for both Filipino and Bisaya; VTLN and LDA+MLLT alone had little to no effect. |
| Phoneme set | PS27 (27 monophones) | The thesis found PS27 slightly better than PS35 -- PS35 sometimes introduced redundant transcriptions. |
| N-gram order | 2-gram | The thesis found 2-gram performed best specifically for Bisaya (3-gram was best for Filipino). |
| HMM states | 3-state | The thesis's best Bisaya HMM-GMM result used 3-state topology; state count showed no consistent trend across its other experiments. |
| CMVN scope | Per-speaker | Kaldi's own default; no override needed. |
| Speed perturbation | Standard Kaldi 3-way (0.9x/1.0x/1.1x), training split only | Standard Kaldi data-augmentation practice. |
| Train/test split | Fixed seed 42, speaker-independent, ~80/20 | Reproducibility, and prevents speaker leakage between train and test. |

The rest of this notebook implements these choices; later sections point
back to this table instead of repeating the reasoning.


In [1]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd):
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                                 executable="/bin/bash", timeout=15)
        return result.stdout.strip()
    except Exception as e:
        return "(error: " + str(e) + ")"

home = str(Path.home())
WORK_DIR = Path(os.environ.get("BISAYA_WORK_DIR", home + "/bisaya_asr")).resolve()
KALDI_ROOT = Path(os.environ.get("KALDI_ROOT", home + "/kaldi")).resolve()
CORPUS_DIR = Path(os.environ.get("BISAYA_CORPUS_DIR", "../data/bisaya_audio")).resolve()
RESULTS_DIR = Path(os.environ.get("BISAYA_RESULTS_DIR", "../output")).resolve()
DATA_ROOT = WORK_DIR / "data"
MFCC_ROOT = WORK_DIR / "mfcc"
EXP_ROOT = WORK_DIR / "exp"
N_JOBS = min(8, os.cpu_count() or 4)

bar = "=" * 78

print(bar)
print("CONFIG")
print(bar)
print("WORK_DIR      =", WORK_DIR)
print("KALDI_ROOT    =", KALDI_ROOT)
corpus_note = "" if CORPUS_DIR.exists() else "  [MISSING]"
print("CORPUS_DIR    =", CORPUS_DIR, corpus_note)
print("RESULTS_DIR   =", RESULTS_DIR)
print("N_JOBS        =", N_JOBS, " (os.cpu_count() =", os.cpu_count(), ")")
print("ALIGN_BEAM / ALIGN_RETRY_BEAM = 40 / 160  (set in Section 13; not re-derivable")
print("  here standalone, this is just what the notebook currently uses)")

def check(label, ok, detail=None):
    mark = "[OK]  " if ok else "[--]  "
    line = mark + label
    if detail:
        line = line + "  -- " + detail
    print(line)

print()
print(bar)
print("KALDI SETUP")
print(bar)
canaries = [
    "tools/extras/install_kenlm_query_only.sh",
    "egs/wsj/s5/steps",
    "egs/wsj/s5/utils",
    "src/Makefile",
]
kaldi_complete = True
for c in canaries:
    if not (KALDI_ROOT / c).exists():
        kaldi_complete = False
check("Kaldi checkout looks complete", kaldi_complete, str(KALDI_ROOT))
check("tools/ built (OpenFST)", (KALDI_ROOT / "tools" / "openfst" / "lib").exists())
check("OpenBLAS built", (KALDI_ROOT / "tools" / "OpenBLAS" / "install" / "lib" / "libopenblas.so").exists())
lmplz_system = run_cmd("which lmplz")
lmplz_local = KALDI_ROOT / "tools" / "kenlm" / "build" / "bin" / "lmplz"
lmplz_ok = bool(lmplz_system) or lmplz_local.exists()
lmplz_detail = lmplz_system if lmplz_system else str(lmplz_local)
check("lmplz (KenLM) available", lmplz_ok, lmplz_detail)
check("Kaldi src/ built (compute-mfcc-feats)",
      (KALDI_ROOT / "src" / "featbin" / "compute-mfcc-feats").exists())
check("conf/mfcc.conf present", (WORK_DIR / "conf" / "mfcc.conf").exists())
check("steps/ and utils/ symlinked", (WORK_DIR / "utils" / "utt2spk_to_spk2utt.pl").exists())

print()
print(bar)
print("DATA PIPELINE")
print(bar)
check("train data dir (wav.scp)", (DATA_ROOT / "train" / "wav.scp").exists())
check("test data dir (wav.scp)", (DATA_ROOT / "test" / "wav.scp").exists())
train_sp_scp = DATA_ROOT / "train_sp" / "wav.scp"
train_sp_detail = None
if train_sp_scp.exists():
    count = run_cmd("wc -l < " + str(train_sp_scp))
    train_sp_detail = count + " utterances"
check("speed-perturbed train_sp", train_sp_scp.exists(), train_sp_detail)
check("lexicon built", (DATA_ROOT / "local" / "dict" / "lexicon.txt").exists())
check("lang/ prepared (L.fst)", (DATA_ROOT / "lang" / "L.fst").exists())
check("2-gram LM (lang_2g/G.fst)", (DATA_ROOT / "lang_2g" / "G.fst").exists())

train_feats = DATA_ROOT / "train_sp" / "feats.scp"
test_feats = DATA_ROOT / "test" / "feats.scp"
check("MFCC+CMVN (train_sp)", train_feats.exists() and train_feats.stat().st_size > 0)
check("MFCC+CMVN (test)", test_feats.exists() and test_feats.stat().st_size > 0)

print()
print(bar)
print("GMM TRAINING")
print(bar)
gmm_stages = [
    ("monophone (mono)", "mono"),
    ("triphone (tri1)", "tri1"),
    ("LDA+MLLT (tri2)", "tri2"),
    ("SAT (tri3, final model)", "tri3"),
]
for stage_name, stage_dir in gmm_stages:
    final_mdl = EXP_ROOT / stage_dir / "final.mdl"
    detail = None
    stage_path = EXP_ROOT / stage_dir
    if not final_mdl.exists() and stage_path.exists():
        n_mdls = run_cmd("ls " + str(stage_path) + "/*.mdl 2>/dev/null | wc -l")
        detail = "in progress -- " + n_mdls + " intermediate .mdl file(s) so far"
    check(stage_name, final_mdl.exists(), detail)

print()
print(bar)
print("DECODE / RESULTS")
print(bar)
graph = EXP_ROOT / "tri3" / "graph" / "HCLG.fst"
check("decode graph (mkgraph)", graph.exists())
decode_dir = EXP_ROOT / "tri3" / "decode_test"
wer_files = list(decode_dir.glob("wer_*")) if decode_dir.exists() else []
wer_detail = None
if wer_files:
    wer_detail = str(len(wer_files)) + " wer_* files"
check("test set decoded", len(wer_files) > 0, wer_detail)
if wer_files:
    wer_cmd = "cd " + str(WORK_DIR) + " && grep WER " + str(decode_dir) + "/wer_* | utils/best_wer.sh"
    best_wer = run_cmd(wer_cmd)
    if not best_wer:
        best_wer = "(could not compute)"
    print("          Best WER so far:", best_wer)
final_results = RESULTS_DIR / "tri3" / "final.mdl"
check("final results copied to Windows (RESULTS_DIR)", final_results.exists(), str(RESULTS_DIR / "tri3"))

print()
print(bar)
print("SYSTEM / LIVE STATE")
print(bar)
print(run_cmd("free -h"))
print()
print(run_cmd("df -h " + str(WORK_DIR) + " 2>/dev/null | tail -1"))
print()
pattern = "gmm-align|gmm-est|train_mono|train_deltas|train_lda_mllt|train_sat|decode_fmllr|lmplz|cmake"
running = run_cmd("ps aux | grep -E '" + pattern + "' | grep -v grep")
if running:
    print("Kaldi-related processes CURRENTLY RUNNING:")
    print(running)
else:
    print("No Kaldi-related processes currently running.")

CONFIG
WORK_DIR      = /home/troxyz1268/bisaya_asr
KALDI_ROOT    = /home/troxyz1268/kaldi
CORPUS_DIR    = /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/data/bisaya_audio 
RESULTS_DIR   = /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/output
N_JOBS        = 8  (os.cpu_count() = 8 )
ALIGN_BEAM / ALIGN_RETRY_BEAM = 40 / 160  (set in Section 13; not re-derivable
  here standalone, this is just what the notebook currently uses)

KALDI SETUP
[OK]  Kaldi checkout looks complete  -- /home/troxyz1268/kaldi
[OK]  tools/ built (OpenFST)
[OK]  OpenBLAS built
[OK]  lmplz (KenLM) available  -- /home/troxyz1268/kaldi/tools/kenlm/build/bin/lmplz
[OK]  Kaldi src/ built (compute-mfcc-feats)
[OK]  conf/mfcc.conf present
[OK]  steps/ and utils/ symlinked

DATA PIPELINE
[OK]  train data dir (wav.scp)
[OK]  test data dir (wav.scp)
[OK]  speed-perturbed train_sp  -- 213 utterances
[OK]  lexicon built
[OK]  lang/ prepared (L.fst)
[OK]  2-gram LM (lang_2g/G.fst)
[OK]  MFCC+CMVN (train_sp)
[O

## 2. Configuration

`CORPUS_DIR` (Parquet corpus, not pre-split into train/test -- the split
happens in Section 8) and `RESULTS_DIR` (final outputs) default to
Windows-hosted relative paths next to this notebook. `WORK_DIR` defaults
to the WSL-native filesystem under `$HOME` for performance (see the intro
cell). All three are overridable via environment variables.


In [2]:
import os
import subprocess
from pathlib import Path

# WORK_DIR: WSL-native filesystem (not /mnt/...) -- Kaldi's build/training
# stages create very large numbers of small files, much slower over the
# Windows bind mount. Only the final model/decode results (Section 15) are
# copied back out to Windows.
WORK_DIR = Path(os.environ.get("BISAYA_WORK_DIR", str(Path.home() / "bisaya_asr"))).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Kaldi is assumed already cloned -- this notebook never runs `git clone`.
KALDI_ROOT = Path(os.environ.get("KALDI_ROOT", str(Path.home() / "kaldi"))).resolve()
assert KALDI_ROOT.exists(), (
    f"KALDI_ROOT={KALDI_ROOT} does not exist -- set the KALDI_ROOT "
    f"environment variable to an existing Kaldi checkout."
)

DATA_ROOT = WORK_DIR / "data"
MFCC_ROOT = WORK_DIR / "mfcc"
EXP_ROOT = WORK_DIR / "exp"

# Dataset stays on Windows, read through WSL's /mnt/... bind mount --
# a handful of large sequential Parquet reads, not the many-small-file
# pattern that's slow over that mount.
CORPUS_DIR = Path(os.environ.get("BISAYA_CORPUS_DIR", "../data/bisaya_audio")).resolve()

# Final artifacts (trained model, decode graph, WER results -- Section 15)
# are copied back to a Windows-hosted directory here.
RESULTS_DIR = Path(os.environ.get("BISAYA_RESULTS_DIR", "../output")).resolve()

print(f"WORK_DIR    = {WORK_DIR}")
print(f"KALDI_ROOT  = {KALDI_ROOT}")
print(f"CORPUS_DIR  = {CORPUS_DIR}")
print(f"RESULTS_DIR = {RESULTS_DIR}")
if not CORPUS_DIR.exists():
    print(f"WARNING: CORPUS_DIR does not exist -- set BISAYA_CORPUS_DIR to "
          f"your actual dataset path before running Section 7.")


WORK_DIR    = /home/troxyz1268/bisaya_asr
KALDI_ROOT  = /home/troxyz1268/kaldi
CORPUS_DIR  = /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/data/bisaya_audio
RESULTS_DIR = /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/output


In [3]:
def sh(cmd, cwd=None, check=True, env=None):
    # Streams output live -- used for every Kaldi binary/script invocation.
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, cwd=cwd, check=check, executable="/bin/bash", env=env)
    return proc.returncode


def stage(name, done, fn):
    # Resumability helper used by every expensive step below: `done` is a
    # zero-arg callable checking whether this stage's output already
    # exists; `fn` does the work otherwise.
    if done():
        print(f"[skip] {name}: output already exists")
        return
    print(f"[run]  {name}")
    fn()


## 3. Install Dependencies

Ubuntu package installation via `apt-get`, which needs `sudo`. If this
cell hangs, run the same `apt-get install` line once from a plain WSL
terminal first (so the password prompt has a TTY to write to), then
re-run this cell -- it will find everything installed and skip through.

Skipped entirely on a re-run once `.deps_installed` exists under
`WORK_DIR`.


In [4]:
import shutil

DEPS_MARKER = WORK_DIR / ".deps_installed"

def _install_apt_deps():
    #sh("sudo apt-get update -qq && sudo apt-get install -y -qq "
    #   "build-essential automake autoconf libtool subversion git zlib1g-dev "
    #   "gfortran sox perl cmake libboost-all-dev libeigen3-dev")
    DEPS_MARKER.touch()

stage("apt dependencies", DEPS_MARKER.exists, _install_apt_deps)

# Checked independently of the marker above so a resumed WORK_DIR still
# gets each one even if the main install was skipped entirely.
stage(
    "install perl (if missing)",
    lambda: shutil.which("perl") is not None,
    lambda: sh("sudo apt-get install -y -qq perl"),
)

# Checked via absolute apt path, not shutil.which -- a pip-installed cmake
# earlier on PATH would otherwise be mistaken for this stage being done.
stage(
    "install cmake (if missing)",
    lambda: Path("/usr/bin/cmake").exists(),
    lambda: sh("sudo apt-get install -y -qq cmake"),
)

# Needed for KenLM's own CMake build (Section 4) to produce lmplz.
stage(
    "install boost + eigen (if missing)",
    lambda: Path("/usr/include/boost/version.hpp").exists(),
    lambda: sh("sudo apt-get install -y -qq libboost-all-dev libeigen3-dev"),
)

# Some Ubuntu images ship only python3, no bare `python` -- needed by a
# diagnostic script decode_fmllr.sh (Section 14) shells out to.
stage(
    "install python-is-python3 (if missing)",
    lambda: shutil.which("python") is not None,
    lambda: sh("sudo apt-get install -y -qq python-is-python3"),
)


def _pip_deps_present():
    try:
        import pandas, pyarrow, tqdm  # noqa: F401
        return True
    except ImportError:
        return False

stage(
    "pip dependencies (pandas, pyarrow, tqdm)",
    _pip_deps_present,
    lambda: sh("pip install -q pandas pyarrow tqdm"),
)


[skip] apt dependencies: output already exists
[skip] install perl (if missing): output already exists
[skip] install cmake (if missing): output already exists
[skip] install boost + eigen (if missing): output already exists
[skip] install python-is-python3 (if missing): output already exists
[skip] pip dependencies (pandas, pyarrow, tqdm): output already exists


## 4. Kaldi: Verify Existing Checkout, Build Tools + KenLM

`KALDI_ROOT` is assumed already cloned. `tools/` (OpenFST etc.) and KenLM
are each built only if their output isn't already present.


In [5]:
def _kaldi_checkout_complete():
    canaries = [
        KALDI_ROOT / "tools" / "extras" / "install_kenlm_query_only.sh",
        KALDI_ROOT / "egs" / "wsj" / "s5" / "steps",
        KALDI_ROOT / "egs" / "wsj" / "s5" / "utils",
        KALDI_ROOT / "src" / "Makefile",
    ]
    return all(c.exists() for c in canaries)

assert _kaldi_checkout_complete(), (
    f"{KALDI_ROOT} doesn't look like a complete Kaldi checkout -- fix or "
    f"re-clone it, or point KALDI_ROOT at a complete one."
)
print(f"Using existing Kaldi checkout at {KALDI_ROOT}")


# Capped below os.cpu_count(): Kaldi/KenLM compile heavy C++ translation
# units and can OOM a memory-constrained VM at higher parallelism. Lower
# to 1 if 2 still causes memory pressure -- make resumes safely either way.
MAKE_JOBS = 2

def _build_tools():
    sh(f"make -j{MAKE_JOBS} CCFLAGS=-std=gnu17", cwd=KALDI_ROOT / "tools")

stage(
    "build Kaldi tools (OpenFST etc.)",
    lambda: (KALDI_ROOT / "tools" / "openfst" / "lib").exists(),
    _build_tools,
)

def _openblas_root():
    return KALDI_ROOT / "tools" / "OpenBLAS" / "install"


def _install_openblas():
    # Stock install_openblas.sh fails to compile under newer GCC's
    # default-error diagnostics; the download/extract step is allowed to
    # "fail" (check=False) and the build+install is redone here with the
    # warning flags relaxed.
    sh("extras/install_openblas.sh", cwd=KALDI_ROOT / "tools", check=False)
    openblas_src = KALDI_ROOT / "tools" / "OpenBLAS"
    cc_override = (
        "gcc -Wno-error=implicit-function-declaration "
        "-Wno-error=incompatible-pointer-types -Wno-error=int-conversion"
    )
    sh(f'make CC="{cc_override}" PREFIX={openblas_src}/install '
       f'USE_LOCKING=1 USE_THREAD=0 -C {openblas_src} all install')

stage(
    "build OpenBLAS (Kaldi's math library)",
    lambda: (_openblas_root() / "lib" / "libopenblas.so").exists(),
    _install_openblas,
)

def _find_lmplz():
    # Prefer an already-installed system lmplz over building a redundant copy.
    system_lmplz = shutil.which("lmplz")
    if system_lmplz:
        return Path(system_lmplz)
    local_lmplz = KALDI_ROOT / "tools" / "kenlm" / "build" / "bin" / "lmplz"
    return local_lmplz if local_lmplz.exists() else None


def _install_kenlm():
    # Kaldi's own install_kenlm_query_only.sh deliberately doesn't build
    # lmplz, so KenLM is built directly here instead (Section 11 needs it).
    kenlm_dir = KALDI_ROOT / "tools" / "kenlm"
    sh(f"rm -rf {kenlm_dir}")
    sh(f"git clone https://github.com/kpu/kenlm.git {kenlm_dir}")

    # This Ubuntu's Boost no longer ships a compiled Boost.System; KenLM
    # doesn't actually need it, so it's dropped from the CMake component list.
    sh(f"sed -i '/^  system$/d' {kenlm_dir}/CMakeLists.txt")

    sh("mkdir -p build && cd build && "
       "/usr/bin/cmake -DBoost_NO_BOOST_CMAKE=ON .. && "
       f"make -j{MAKE_JOBS}", cwd=kenlm_dir)

stage(
    "install KenLM (full build, for lmplz)",
    lambda: _find_lmplz() is not None,
    _install_kenlm,
)


Using existing Kaldi checkout at /home/troxyz1268/kaldi
[skip] build Kaldi tools (OpenFST etc.): output already exists
[skip] build OpenBLAS (Kaldi's math library): output already exists
[skip] install KenLM (full build, for lmplz): output already exists


## 5. Kaldi Recipe: Build Kaldi Binaries (`src/`)

Skipped entirely if `compute-mfcc-feats` already exists -- the slowest
step in the notebook (30-90 minutes cold).


In [6]:
src_dir = KALDI_ROOT / "src"
mfcc_bin = src_dir / "featbin" / "compute-mfcc-feats"

# --use-cuda=no targets CPU-only environments; drop it if you have a
# CUDA-capable GPU you want Kaldi to use.
def _build_kaldi_src():
    if not (src_dir / "kaldi.mk").exists():
        sh(f"./configure --shared --use-cuda=no --mathlib=OPENBLAS "
           f"--openblas-root={_openblas_root()}", cwd=src_dir)
    sh(f"make -j{MAKE_JOBS} depend", cwd=src_dir)
    sh(f"make -j{MAKE_JOBS}", cwd=src_dir)

stage("build Kaldi (src/)", mfcc_bin.exists, _build_kaldi_src)

print("Kaldi build complete." if mfcc_bin.exists()
      else "WARNING: expected binary not found -- check the build log above for errors.")


[skip] build Kaldi (src/): output already exists
Kaldi build complete.


## 6. Kaldi Recipe Scaffolding (`path.sh`, `cmd.sh`, `steps/`, `utils/`)


In [7]:
path_sh_lines = [
    f"export KALDI_ROOT={KALDI_ROOT}",
    "export PATH=$PWD/utils/:$KALDI_ROOT/tools/openfst/bin:$PWD:$PATH",
    ". $KALDI_ROOT/tools/config/common_path.sh",
    "export LC_ALL=C",
]
(WORK_DIR / "path.sh").write_text("\n".join(path_sh_lines) + "\n")

cmd_sh_lines = [
    "export train_cmd=run.pl",
    "export decode_cmd=run.pl",
    "export mkgraph_cmd=run.pl",
]
(WORK_DIR / "cmd.sh").write_text("\n".join(cmd_sh_lines) + "\n")

wsj_s5 = KALDI_ROOT / "egs" / "wsj" / "s5"
for name in ("steps", "utils"):
    link = WORK_DIR / name
    if not link.exists():
        link.symlink_to(wsj_s5 / name)

# local/score.sh: every egs/*/s5 recipe symlinks this to steps/score_kaldi.sh;
# without it, decode_fmllr.sh (Section 14) silently skips scoring.
(WORK_DIR / "local").mkdir(exist_ok=True)
score_sh = WORK_DIR / "local" / "score.sh"
if not score_sh.exists():
    score_sh.symlink_to(Path("../steps/score_kaldi.sh"))

# conf/mfcc.conf: steps/make_mfcc.sh (Section 12) requires this by default;
# WORK_DIR isn't a real recipe dir, so it's created explicitly here.
(WORK_DIR / "conf").mkdir(exist_ok=True)
mfcc_conf = WORK_DIR / "conf" / "mfcc.conf"
if not mfcc_conf.exists():
    mfcc_conf.write_text("--use-energy=false   # only non-default option.\n")

probe = WORK_DIR / "utils" / "utt2spk_to_spk2utt.pl"
assert probe.exists(), (
    f"{probe} not found -- steps/utils symlinks are broken (check that "
    f"{wsj_s5} exists and KALDI_ROOT={KALDI_ROOT} is correct)."
)

print("path.sh, cmd.sh, steps/, utils/ ready under", WORK_DIR)


path.sh, cmd.sh, steps/, utils/ ready under /home/troxyz1268/bisaya_asr


## 7. Data Loading

Reads every Parquet shard in `CORPUS_DIR` into pandas and concatenates in
memory. Each shard is read exactly once, in sorted-filename order --
Section 8's utterance IDs depend on this exact ordering.


In [8]:
import pandas as pd

def load_all_shards(corpus_dir):
    files = sorted(Path(corpus_dir).glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    dfs = []
    for file in files:
        print(f"Loading: {file.name}")
        temp = pd.read_parquet(file)
        temp["source_file"] = file.name
        dfs.append(temp)
    return pd.concat(dfs, ignore_index=True)


corpus_df = load_all_shards(CORPUS_DIR)
print(f"\nLoaded {len(corpus_df):,} utterances from {CORPUS_DIR}")
print(f"Speakers: {corpus_df['speaker_id'].nunique() if len(corpus_df) else 0:,}")

Loading: test-00000-of-00011.parquet
Loading: test-00001-of-00011.parquet
Loading: test-00002-of-00011.parquet
Loading: test-00003-of-00011.parquet
Loading: test-00004-of-00011.parquet
Loading: test-00005-of-00011.parquet
Loading: test-00006-of-00011.parquet
Loading: test-00007-of-00011.parquet
Loading: test-00008-of-00011.parquet
Loading: test-00009-of-00011.parquet
Loading: test-00010-of-00011.parquet

Loaded 90 utterances from /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/data/bisaya_audio
Speakers: 15


## 8. Speaker-Independent Train/Test Split + Kaldi Data Dirs

Split **by speaker**, not utterance, ~80/20, fixed seed (see Section 1).
`build_kaldi_data_dir()` writes each utterance's audio to its own `.wav`
plus `wav.scp`/`text`/`utt2spk`; skipped per split if already done.


In [9]:
import re

SPLIT_SEED = 42  # fixed and printed so this split is reproducible on rerun

def speaker_independent_split(df, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(df["speaker_id"].unique())
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])

    assert train_speakers.isdisjoint(test_speakers)

    return df[df["speaker_id"].isin(train_speakers)].copy(), df[df["speaker_id"].isin(test_speakers)].copy()


train_df, test_df = speaker_independent_split(corpus_df)

print(f"seed = {SPLIT_SEED}")
print(f"train: {train_df['speaker_id'].nunique()} speakers, {len(train_df)} utterances")
print(f"test:  {test_df['speaker_id'].nunique()} speakers, {len(test_df)} utterances")
print(f"speaker overlap: {set(train_df['speaker_id']) & set(test_df['speaker_id'])}")


def kaldi_normalize_text(text):
    # Lowercase + strip punctuation only -- does NOT fold u/o the way the
    # evaluation notebooks' WER-scoring normalization does; that's a
    # scoring-time leniency, not a valid training transcript.
    text = text.lower()
    text = re.sub(r"[^\w\s']", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_kaldi_data_dir(df, out_dir, wav_out_dir):
    out_dir = Path(out_dir)
    wav_out_dir = Path(wav_out_dir)

    if (out_dir / "wav.scp").exists():
        print(f"[skip] {out_dir}: wav.scp already exists")
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    wav_out_dir.mkdir(parents=True, exist_ok=True)

    wav_lines, text_lines, utt2spk_lines = [], [], []

    for i, row in df.iterrows():
        # utt-ids are prefixed with speaker-id, required for utt2spk/spk2utt
        # sort-order conventions -- and encode the row's corpus index,
        # which evaluate_kaldi.ipynb later uses to look up metadata.
        spk = str(row["speaker_id"])
        utt_id = f"{spk}-{i:06d}"

        wav_path = wav_out_dir / f"{utt_id}.wav"
        wav_path.write_bytes(row["audio"]["bytes"])

        wav_lines.append(f"{utt_id} sox {wav_path} -r 16000 -c 1 -t wav - |")

        transcript = kaldi_normalize_text(str(row["transcript"]))
        text_lines.append(f"{utt_id} {transcript}")
        utt2spk_lines.append(f"{utt_id} {spk}")

    (out_dir / "wav.scp").write_text("\n".join(sorted(wav_lines)) + "\n")
    (out_dir / "text").write_text("\n".join(sorted(text_lines)) + "\n")
    (out_dir / "utt2spk").write_text("\n".join(sorted(utt2spk_lines)) + "\n")

    sh(f"utils/utt2spk_to_spk2utt.pl {out_dir}/utt2spk > {out_dir}/spk2utt", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {out_dir}", cwd=WORK_DIR)
    sh(f"utils/validate_data_dir.sh --no-feats --non-print {out_dir}", cwd=WORK_DIR)

    print(f"{out_dir}: {len(wav_lines)} utterances, {df['speaker_id'].nunique()} speakers")


build_kaldi_data_dir(train_df, DATA_ROOT / "train", WORK_DIR / "wav" / "train")
build_kaldi_data_dir(test_df, DATA_ROOT / "test", WORK_DIR / "wav" / "test")


seed = 42
train: 12 speakers, 71 utterances
test:  3 speakers, 19 utterances
speaker overlap: set()
[skip] /home/troxyz1268/bisaya_asr/data/train: wav.scp already exists
[skip] /home/troxyz1268/bisaya_asr/data/test: wav.scp already exists


In [10]:
# corpus_df/train_df/test_df hold the full corpus's raw audio bytes in
# memory; every utterance is already written to its own .wav file above,
# so these are dead weight that can otherwise starve later stages of RAM.
del corpus_df, train_df, test_df
import gc
gc.collect()


0

## 9. Speed Perturbation (Training Data Only)

Kaldi's standard 3-way speed perturbation (0.9x/1.0x/1.1x), applied only to
the training set -- see Section 1. Skipped if `train_sp/wav.scp` already
exists.


In [11]:
TRAIN_DATA_DIR = DATA_ROOT / "train_sp"

def _speed_perturb():
    sh(f"utils/data/perturb_data_dir_speed_3way.sh {DATA_ROOT}/train {TRAIN_DATA_DIR}", cwd=WORK_DIR)
    sh(f"utils/fix_data_dir.sh {TRAIN_DATA_DIR}", cwd=WORK_DIR)

stage(
    "speed perturbation",
    lambda: (TRAIN_DATA_DIR / "wav.scp").exists(),
    _speed_perturb,
)

[skip] speed perturbation: output already exists


## 10. Lexicon and Phoneme Data (`data/local/dict`, PS27)

Builds a lexicon constrained to **PS27**'s 27-monophone inventory -- see
Section 1 for why PS27 over PS35. Lexicon build and `prepare_lang.sh` are
each skipped if their own output already exists.


In [12]:
PS27_PHONES = {
    "p", "b", "t", "d", "k", "g",
    "f", "v", "s", "z", "sh", "th", "h",
    "j", "ch",
    "m", "n", "ng",
    "l", "r",
    "w", "y",
    "a", "e", "i", "o", "u",
}


def word_to_phones(word):
    w = re.sub(r"[^a-z']", "", word.lower()).replace("'", "")
    phones = []
    i = 0
    while i < len(w):
        if w[i:i + 2] == "ng":
            phones.append("ng")
            i += 2
        elif w[i:i + 2] == "sh":
            phones.append("sh")
            i += 2
        elif w[i:i + 2] == "th":
            phones.append("th")
            i += 2
        elif w[i:i + 2] in ("ts", "ty"):
            phones.append("ch")
            i += 2
        elif w[i] == "c":
            phones.append("s" if w[i + 1:i + 2] in ("e", "i") else "k")
            i += 1
        elif w[i] == "q":
            phones.append("k")
            i += 1
        elif w[i] == "x":
            phones.extend(["k", "s"])
            i += 1
        else:
            phones.append(w[i])
            i += 1

    assert all(p in PS27_PHONES for p in phones), f"{word} -> {phones} outside PS27"
    return phones


def build_lexicon(text_paths, dict_dir):
    dict_dir = Path(dict_dir)

    if (dict_dir / "lexicon.txt").exists():
        print(f"[skip] {dict_dir}: lexicon.txt already exists")
        return dict_dir

    dict_dir.mkdir(parents=True, exist_ok=True)

    vocab = set()
    for p in text_paths:
        for line in Path(p).read_text().splitlines():
            words = line.split(" ")[1:]
            vocab.update(w for w in words if w)

    lexicon_lines = ["<unk> spn"]
    all_phones = set()
    for word in sorted(vocab):
        phones = word_to_phones(word)
        if not phones:
            continue
        all_phones.update(phones)
        lexicon_lines.append(f"{word} {' '.join(phones)}")

    (dict_dir / "lexicon.txt").write_text("\n".join(lexicon_lines) + "\n")
    (dict_dir / "silence_phones.txt").write_text("sil\nspn\n")
    (dict_dir / "optional_silence.txt").write_text("sil\n")
    (dict_dir / "nonsilence_phones.txt").write_text("\n".join(sorted(all_phones)) + "\n")
    (dict_dir / "extra_questions.txt").write_text("")

    print(f"Vocabulary: {len(vocab)} words, {len(all_phones)} distinct phones (of PS27's 27) -> {dict_dir}")
    return dict_dir


LOCAL_DICT_DIR = build_lexicon(
    [DATA_ROOT / "train" / "text", DATA_ROOT / "test" / "text"],
    DATA_ROOT / "local" / "dict",
)

LANG_DIR = DATA_ROOT / "lang"

def _prepare_lang():
    sh(f"utils/prepare_lang.sh --position-dependent-phones false "
       f"{LOCAL_DICT_DIR} '<unk>' {DATA_ROOT}/local/lang {LANG_DIR}", cwd=WORK_DIR)

stage("prepare_lang", lambda: (LANG_DIR / "L.fst").exists(), _prepare_lang)

[skip] /home/troxyz1268/bisaya_asr/data/local/dict: lexicon.txt already exists
[skip] prepare_lang: output already exists


## 11. Language Model (KenLM -> ARPA -> Kaldi `G.fst`)

Builds a **2-gram** word-level LM from the training transcripts -- see
Section 1 for why 2-gram (the thesis found it best specifically for
Bisaya, unlike Filipino where 3-gram won). Skipped if `lang_2g/G.fst`
already exists. Uses whichever `lmplz` Section 4 found or built
(`_find_lmplz()`), so this doesn't care whether that came from a
pre-existing system install or the KenLM build inside the Kaldi checkout.


In [13]:
LM_DIR = DATA_ROOT / "local" / "lm"
LANG_2G_DIR = DATA_ROOT / "lang_2g"

def _build_lm():
    lmplz = _find_lmplz()
    assert lmplz is not None, "lmplz not found -- Section 4's KenLM install stage should have built it."
    LM_DIR.mkdir(parents=True, exist_ok=True)

    train_text_lines = (DATA_ROOT / "train" / "text").read_text().splitlines()
    corpus_txt = LM_DIR / "corpus.txt"
    corpus_txt.write_text("\n".join(" ".join(line.split(" ")[1:]) for line in train_text_lines) + "\n")

    arpa_path = LM_DIR / "2gram.arpa"
    sh(f"{lmplz} -o 2 --discount_fallback < {corpus_txt} > {arpa_path}")

    # format_lm.sh expects a gzipped ARPA file; lmplz writes plain text.
    arpa_gz_path = Path(f"{arpa_path}.gz")
    sh(f"gzip -kf {arpa_path}")

    sh(f"utils/format_lm.sh {LANG_DIR} {arpa_gz_path} {LOCAL_DICT_DIR}/lexicon.txt "
       f"{LANG_2G_DIR}", cwd=WORK_DIR)

stage("build 2-gram LM", lambda: (LANG_2G_DIR / "G.fst").exists(), _build_lm)

print("lang_2g graph directory:", LANG_2G_DIR)


[skip] build 2-gram LM: output already exists
lang_2g graph directory: /home/troxyz1268/bisaya_asr/data/lang_2g


## 12. MFCC + CMVN Feature Extraction

25 ms window, 10 ms frameshift, 13 static coefficients -- Kaldi's MFCC
defaults already match this. CMVN is per-speaker (see Section 1). Skipped
per split if that split's `feats.scp` already exists and is non-empty.


In [14]:
N_JOBS = min(8, os.cpu_count() or 4)

def _make_extract_fn(name, data_dir):
    def _fn():
        sh(f"steps/make_mfcc.sh --cmd run.pl --nj {N_JOBS} {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
        sh(f"steps/compute_cmvn_stats.sh {data_dir} {EXP_ROOT}/make_mfcc/{name} {MFCC_ROOT}", cwd=WORK_DIR)
        sh(f"utils/fix_data_dir.sh {data_dir}", cwd=WORK_DIR)
    return _fn


for name, data_dir in [("train", TRAIN_DATA_DIR), ("test", DATA_ROOT / "test")]:
    feats_scp = Path(data_dir) / "feats.scp"
    stage(
        f"MFCC+CMVN ({name})",
        lambda p=feats_scp: p.exists() and p.stat().st_size > 0,
        _make_extract_fn(name, data_dir),
    )

[skip] MFCC+CMVN (train): output already exists
[skip] MFCC+CMVN (test): output already exists


## 13. HMM-GMM Training (Monophone -> Triphone -> LDA+MLLT -> SAT)

The standard Kaldi progression: monophone, then triphone (delta features),
then LDA+MLLT, then speaker-adaptive training (SAT) -- SAT is this
notebook's **final model** (see Section 1). Each stage is skipped if its
own `final.mdl` already exists.


In [15]:
# Alignment/retry beams widened well past Kaldi's WSJ-recipe defaults
# (tuned for short read-speech sentences). This corpus's utterances average
# ~105s and run up to ~330s, which caused most monophone alignments to fail
# at the defaults. Widening recovers the large majority; a residual handful
# of long, code-switched utterances still fail regardless of beam width
# (a lexicon-coverage limit, not a beam-width one).
ALIGN_BEAM = 40
ALIGN_RETRY_BEAM = 160


def _train_mono():
    sh(f"steps/train_mono.sh --cmd run.pl --nj {N_JOBS} "
       f"--initial-beam {ALIGN_BEAM} --regular-beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono {EXP_ROOT}/mono_ali", cwd=WORK_DIR)

stage("monophone (mono)", lambda: (EXP_ROOT / "mono" / "final.mdl").exists(), _train_mono)


def _train_tri1():
    sh(f"steps/train_deltas.sh --cmd run.pl "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} 2000 10000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/mono_ali {EXP_ROOT}/tri1", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri1 {EXP_ROOT}/tri1_ali", cwd=WORK_DIR)

stage("triphone (tri1, delta features)", lambda: (EXP_ROOT / "tri1" / "final.mdl").exists(), _train_tri1)


def _train_tri2():
    sh(f"steps/train_lda_mllt.sh --cmd run.pl "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} 2500 15000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri1_ali {EXP_ROOT}/tri2", cwd=WORK_DIR)
    sh(f"steps/align_si.sh --cmd run.pl --nj {N_JOBS} "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri2 {EXP_ROOT}/tri2_ali", cwd=WORK_DIR)

stage("LDA+MLLT (tri2)", lambda: (EXP_ROOT / "tri2" / "final.mdl").exists(), _train_tri2)


def _train_tri3():
    sh(f"steps/train_sat.sh --cmd run.pl "
       f"--beam {ALIGN_BEAM} --retry-beam {ALIGN_RETRY_BEAM} 2500 15000 "
       f"{TRAIN_DATA_DIR} {LANG_DIR} {EXP_ROOT}/tri2_ali {EXP_ROOT}/tri3", cwd=WORK_DIR)

stage("SAT (tri3) -- final model", lambda: (EXP_ROOT / "tri3" / "final.mdl").exists(), _train_tri3)


[skip] monophone (mono): output already exists
[skip] triphone (tri1, delta features): output already exists
[skip] LDA+MLLT (tri2): output already exists
[skip] SAT (tri3) -- final model: output already exists


## 14. Decode + Evaluate the Final Model (SAT, 2-gram, PS27)

Builds the decode graph from `lang_2g` and the `tri3` model, decodes the
test set, sweeps the LM weight over 1-25, and prints the best WER. Compare
against the thesis's reported 5.41% figure (expect a gap -- different,
much smaller corpus and test set).


In [16]:
GRAPH_DIR = EXP_ROOT / "tri3" / "graph"
DECODE_DIR = EXP_ROOT / "tri3" / "decode_test"

def _mkgraph():
    sh(f"utils/mkgraph.sh {LANG_2G_DIR} {EXP_ROOT}/tri3 {GRAPH_DIR}", cwd=WORK_DIR)

stage("build decode graph (mkgraph)", lambda: (GRAPH_DIR / "HCLG.fst").exists(), _mkgraph)


def _decode():
    # Beams widened for the same long-utterance reason as Section 13.
    # decode_fmllr.sh splits work per-speaker, so --nj can't exceed the
    # test set's speaker count.
    n_test_spk = sum(1 for _ in open(DATA_ROOT / "test" / "spk2utt"))
    decode_nj = max(1, min(N_JOBS, n_test_spk))
    sh(f"steps/decode_fmllr.sh --cmd run.pl --nj {decode_nj} "
       f"--beam 30 --lattice-beam 15 "
       f"--scoring-opts \"--min-lmwt 1 --max-lmwt 25\" "
       f"{GRAPH_DIR} {DATA_ROOT}/test {DECODE_DIR}", cwd=WORK_DIR)

stage(
    "decode test set",
    lambda: DECODE_DIR.exists() and any(DECODE_DIR.glob("wer_*")),
    _decode,
)

sh(f"grep WER {DECODE_DIR}/wer_* | utils/best_wer.sh", cwd=WORK_DIR)


[skip] build decode graph (mkgraph): output already exists
[skip] decode test set: output already exists
$ grep WER /home/troxyz1268/bisaya_asr/exp/tri3/decode_test/wer_* | utils/best_wer.sh
%WER 41.02 [ 2418 / 5894, 380 ins, 273 del, 1765 sub ] /home/troxyz1268/bisaya_asr/exp/tri3/decode_test/wer_24_1.0


0

## 15. Export Final Results and Model

Copies the trained model, decode graph, and WER output (`exp/tri3`) to
`RESULTS_DIR` on Windows, then curates a smaller `models/tri3/` subset
containing just what's needed to decode new audio (not the training-only
alignments/logs) -- see `models/tri3/README.md`.


In [17]:
import shutil

FINAL_MODEL_RESULTS_DIR = RESULTS_DIR / "tri3"

def _copy_results():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        sh(f"rsync -a {EXP_ROOT}/tri3/ {FINAL_MODEL_RESULTS_DIR}/")
    else:
        sh(f"rm -rf {FINAL_MODEL_RESULTS_DIR} && cp -r {EXP_ROOT}/tri3 {FINAL_MODEL_RESULTS_DIR}")

stage(
    "copy final model + decode results to Windows",
    lambda: (FINAL_MODEL_RESULTS_DIR / "final.mdl").exists(),
    _copy_results,
)

print(f"Final model, graph, and decode/WER output available at: {FINAL_MODEL_RESULTS_DIR}")


# Curated, decode-ready subset of the above -- just the files needed to
# decode new audio (final.mdl, final.alimdl, tree, transforms, graph/),
# not training-only artifacts (per-job alignments, FSTs, logs).
MODELS_DIR = Path(os.environ.get("BISAYA_MODELS_DIR", "../models")).resolve() / "tri3"

def _export_model():
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("final.mat", "full.mat", "tree", "cmvn_opts", "splice_opts", "phones.txt"):
        shutil.copy(FINAL_MODEL_RESULTS_DIR / name, MODELS_DIR / name)
    for name in ("final.mdl", "final.alimdl"):
        # Dereference the symlinks (e.g. final.mdl -> 35.mdl) so the
        # curated copy is self-contained.
        shutil.copy(os.path.realpath(FINAL_MODEL_RESULTS_DIR / name), MODELS_DIR / name)
    graph_dst = MODELS_DIR / "graph"
    if graph_dst.exists():
        shutil.rmtree(graph_dst)
    shutil.copytree(FINAL_MODEL_RESULTS_DIR / "graph", graph_dst)
    print(f"Deployable model exported to: {MODELS_DIR}")

stage(
    "export curated model to models/tri3",
    lambda: (MODELS_DIR / "final.mdl").exists(),
    _export_model,
)


[skip] copy final model + decode results to Windows: output already exists
Final model, graph, and decode/WER output available at: /mnt/c/Users/windows 10/Documents/Sugbodoc/speech model/output/tri3
[skip] export curated model to models/tri3: output already exists


## 16. Summary

The Section 14 best-WER line is this notebook's final result: a 2-gram,
3-state, SAT HMM-GMM model using PS27. Compare against the thesis's
5.41% figure, expecting a gap due to corpus differences and Section 10's
lexicon (`word_to_phones()` is a rule-based approximation, not the
thesis's own transcriber-produced PS27 dictionary).

Re-running top to bottom after an interruption is safe -- every stage
skips straight past its own completed output. Final artifacts are under
`RESULTS_DIR/tri3` (full) and `models/tri3` (curated, for
`evaluate_kaldi.ipynb`).
